# RAG Pipeline Fundamentals — Hands-On

Offline numpy demo of chunking, deterministic embeddings, retrieval, context assembly, and citation checking.

## 0. Setup: tiny corpus

In [ ]:
%pip install -q numpy
import hashlib, numpy as np

docs = {"rag":"RAG retrieves external evidence before generation and cites sources.",
        "tune":"Fine tuning teaches behavior, style, and output format, not fresh facts.",
        "long":"Long context works when the full relevant working set fits in the prompt.",
        "cite":"A grounded answer should cite chunk identifiers for factual claims."}
print(len(docs), "documents")

## 1. Chunk and embed

In [ ]:
def embed(text, dim=48):
    v = np.zeros(dim)
    for tok in text.lower().replace('.', '').split():
        h = int(hashlib.sha256(tok.encode()).hexdigest(), 16)
        v[h % dim] += 1
    return v / (np.linalg.norm(v) + 1e-9)

chunks = [(doc_id, text, embed(text)) for doc_id, text in docs.items()]
print(chunks[0][0], np.round(chunks[0][2][:6], 2))

## 2. Retrieve top-k

In [ ]:
def retrieve(q, k=2):
    qv = embed(q)
    scored = [(float(qv @ vec), doc_id, text) for doc_id, text, vec in chunks]
    return sorted(scored, reverse=True)[:k]

hits = retrieve("Should I use RAG for citations and fresh facts?", 3)
for s, i, t in hits: print(f"{s:.3f}", i, t)

## 3. Assemble context with stable citation IDs

In [ ]:
def assemble(hits, budget=260):
    lines, used = [], 0
    for score, doc_id, text in hits:
        line = f"[{doc_id}] {text}"
        if used + len(line) <= budget:
            lines.append(line); used += len(line)
    return "Answer only from the context and cite IDs.\n<context>\n" + "\n".join(lines) + "\n</context>"

ctx = assemble(hits)
print(ctx)

## 4. Simulated generator + citation validation

In [ ]:
def fake_generate(question, context):
    if "[rag]" in context and "[cite]" in context:
        return "Use RAG when answers need fresh external evidence and cite sources [rag][cite]."
    return "insufficient_context"

def cited_ids(answer):
    return [part.split(']')[0] for part in answer.split('[')[1:]]

ans = fake_generate("When use RAG?", ctx)
allowed = {doc_id for _, doc_id, _ in hits}
print(ans)
print("citations valid:", set(cited_ids(ans)).issubset(allowed))

## 5. RAG vs alternatives as a decision rule

In [ ]:
def choose(knowledge_changes, needs_citations, behavior_change, corpus_fits):
    if behavior_change and not needs_citations: return "fine-tune"
    if corpus_fits and not knowledge_changes: return "long-context"
    if knowledge_changes or needs_citations: return "RAG"
    return "prompting"

cases = [(1,1,0,0), (0,0,1,0), (0,0,0,1)]
for c in cases: print(c, "->", choose(*c))

## 6. Exercise prompts
1. Add ACL metadata and filter retrieval.
2. Change chunk size and measure hit rate.
3. Force `insufficient_context` when top score is below a threshold.